In [3]:
from datasets import load_dataset

dataset = load_dataset("davanstrien/WELFake")
print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-290868f0a36350(…):   0%|          | 0.00/152M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/72134 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'text', 'label'],
        num_rows: 72134
    })
})


In [4]:
# Vérifier si test split existe, sinon créer 10% du train pour test
if "test" not in dataset:
    dataset = dataset["train"].train_test_split(test_size=0.1)
    print("\n--- Splits après création du test ---")
    print(dataset)

print(f"\nNombre d'exemples : Train = {len(dataset['train'])}, Test = {len(dataset['test'])}")


--- Splits après création du test ---
DatasetDict({
    train: Dataset({
        features: ['title', 'text', 'label'],
        num_rows: 64920
    })
    test: Dataset({
        features: ['title', 'text', 'label'],
        num_rows: 7214
    })
})

Nombre d'exemples : Train = 64920, Test = 7214


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
import re
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# ======================
# Configuration
# ======================
MODEL_NAME = "roberta-base"
OUTPUT_DIR = "/content/drive/MyDrive/roberta_welfake"
MAX_LENGTH = 512


# ======================
# 2. Prétraitement du texte
# ======================
def clean_text(text):
    """Nettoyage basique du texte avec gestion des valeurs nulles"""
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(batch):
    cleaned_texts = [clean_text(t) for t in batch["text"]]
    tokens = tokenizer(
        cleaned_texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    tokens["labels"] = batch["label"]
    return tokens

# ======================
# 3. Tokenizer
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = dataset.map(preprocess, batched=True)
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# ======================
# 4. Modèle Transformer
# ======================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# ======================
# 5. Fonction metrics
# ======================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# ======================
# 6. Entraînement
# ======================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()

# ======================
# 7. Évaluation
# ======================
print("\n==============================")
print(" ÉVALUATION DU MODÈLE ")
print("==============================")

predictions = trainer.predict(dataset["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix :")
print(cm)

print("\nClassification Report :")
print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))

# ======================
# 8. Sauvegarde du modèle
# ======================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Fine-tuning WELFake terminé avec succès.")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.445900
400,0.263500
600,0.195400
800,0.180400
1000,0.139400
1200,0.166600
1400,0.172700
1600,0.168000
1800,0.186700
2000,0.125400


Step,Training Loss
200,0.445900
400,0.263500
600,0.195400
800,0.180400
1000,0.139400
1200,0.166600
1400,0.172700
1600,0.168000
1800,0.186700
2000,0.125400



 ÉVALUATION DU MODÈLE 


Accuracy  : 0.9882
Precision : 0.9847
Recall    : 0.9928
F1-score  : 0.9887

Confusion Matrix :
[[3400   58]
 [  27 3729]]

Classification Report :
              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      3458
        Real       0.98      0.99      0.99      3756

    accuracy                           0.99      7214
   macro avg       0.99      0.99      0.99      7214
weighted avg       0.99      0.99      0.99      7214

✅ Fine-tuning WELFake terminé avec succès.
